In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!pip install sentence-transformers torch matplotlib scipy scikit-learn -q

# Experiment Overview

Experiment 10 — Grammatical Frame Induction

Motivation:
  Exp 9 Probe 2 found that transitivity (binary: transitive vs intransitive)
  is NOT encoded in the multilingual concept space (LOO acc = chance).
  But transitivity-as-binary is a coarse test. The suggestion is that
  frame ROLES (agent, patient, instrument, theme, experiencer, goal)
  may have continuous geometric structure even though the binary split doesn't.

  This experiment:
  1. Annotates 56 concepts with VerbNet-inspired frame roles
  2. Probes whether frame role structure is recoverable from:
     (a) Raw LaBSE centroids
     (b) Phase 1 projected space (same projection from Exp 9)
  3. Tests a key prediction: if frame roles have geometry, then
     verb-noun COMPOSITION should be predictable — given "give" and "water",
     can we predict which is agent-slot vs patient-slot?

  If roles ARE recoverable: the universal language gets agglutinative
  suffixes (like Esperanto's -i/-as/-is) grounded in data.
  If NOT: frame structure must be hand-designed.

In [ ]:
import os, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.spatial.distance import pdist, squareform, cosine as cos_dist
from scipy.stats import spearmanr
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

model = SentenceTransformer('LaBSE')
print('LaBSE loaded')

## SECTION 1: Frame-annotated concept vocabulary

## Frame roles (VerbNet/FrameNet-inspired): AGENT:       entity that initiates/controls action PATIENT:     entity affected/changed by action THEME:       entity that moves or is located EXPERIENCER: entity that perceives/feels INSTRUMENT:  means by which action is performed GOAL:        endpoint or purpose STATE:       non-dynamic property or condition NONE:        concepts without frame participation (concrete objects, abstract states)

In [ ]:
#
# For verbs: primary_role = the role the verb's SUBJECT typically fills
# For nouns: typical_role = the frame slot this noun typically fills

CONCEPT_DEFS = [
    # concept,        POS,               primary_role,    can_be_agent, can_be_patient, can_be_instrument
    ('water',         'concrete_noun',   'THEME',         False, True,  True),
    ('fire',          'concrete_noun',   'THEME',         False, True,  True),
    ('earth',         'concrete_noun',   'THEME',         False, True,  False),
    ('sky',           'concrete_noun',   'THEME',         False, False, False),
    ('wind',          'concrete_noun',   'THEME',         False, True,  True),
    ('stone',         'concrete_noun',   'THEME',         False, True,  True),
    ('river',         'concrete_noun',   'THEME',         False, False, False),
    ('mountain',      'concrete_noun',   'THEME',         False, False, False),
    ('forest',        'concrete_noun',   'THEME',         False, False, False),
    ('child',         'concrete_noun',   'AGENT',         True,  True,  False),
    ('elder',         'concrete_noun',   'AGENT',         True,  True,  False),
    ('friend',        'concrete_noun',   'AGENT',         True,  True,  False),
    ('enemy',         'concrete_noun',   'AGENT',         True,  True,  False),
    ('time',          'abstract_noun',   'THEME',         False, False, False),
    ('life',          'abstract_noun',   'THEME',         False, False, False),
    ('death',         'abstract_noun',   'STATE',         False, False, False),
    ('peace',         'abstract_noun',   'STATE',         False, False, False),
    ('war',           'abstract_noun',   'STATE',         False, False, False),
    ('hope',          'abstract_noun',   'EXPERIENCER',   False, True,  False),  # "I feel hope" → experiencer-themed
    ('dream',         'abstract_noun',   'THEME',         False, True,  False),
    ('change',        'abstract_noun',   'THEME',         False, True,  False),
    ('love',          'emotion',         'EXPERIENCER',   False, True,  False),
    ('fear',          'emotion',         'EXPERIENCER',   False, True,  False),
    ('trust',         'emotion',         'EXPERIENCER',   False, True,  False),
    ('joy',           'emotion',         'EXPERIENCER',   False, True,  False),
    ('pain',          'emotion',         'EXPERIENCER',   False, True,  False),
    ('give',          'transitive_verb', 'AGENT',         True,  False, False),
    ('take',          'transitive_verb', 'AGENT',         True,  False, False),
    ('speak',         'transitive_verb', 'AGENT',         True,  False, False),
    ('think',         'transitive_verb', 'EXPERIENCER',   True,  False, False),
    ('find',          'transitive_verb', 'AGENT',         True,  False, False),
    ('lose',          'transitive_verb', 'EXPERIENCER',   True,  False, False),
    ('build',         'transitive_verb', 'AGENT',         True,  False, False),
    ('break',         'transitive_verb', 'AGENT',         True,  False, False),
    ('eat',           'transitive_verb', 'AGENT',         True,  False, False),
    ('feel',          'transitive_verb', 'EXPERIENCER',   True,  False, False),
    ('sleep',         'intransitive_verb','EXPERIENCER',  True,  False, False),
    ('run',           'intransitive_verb','AGENT',        True,  False, False),
    ('light',         'concrete_noun',   'THEME',         False, True,  True),
    ('dark',          'abstract_noun',   'STATE',         False, False, False),
    # Extended set for richer frame analysis (16 more concepts)
    ('teach',         'transitive_verb', 'AGENT',         True,  False, False),
    ('learn',         'transitive_verb', 'EXPERIENCER',   True,  False, False),
    ('kill',          'transitive_verb', 'AGENT',         True,  False, False),
    ('grow',          'intransitive_verb','THEME',        True,  True,  False),
    ('fall',          'intransitive_verb','THEME',        False, False, False),
    ('rise',          'intransitive_verb','THEME',        False, False, False),
    ('sing',          'intransitive_verb','AGENT',        True,  False, False),
    ('cry',           'intransitive_verb','EXPERIENCER',  True,  False, False),
    ('knife',         'concrete_noun',   'INSTRUMENT',    False, False, True),
    ('hand',          'concrete_noun',   'INSTRUMENT',    False, True,  True),
    ('voice',         'concrete_noun',   'INSTRUMENT',    False, False, True),
    ('home',          'concrete_noun',   'GOAL',          False, False, False),
    ('silence',       'abstract_noun',   'STATE',         False, False, False),
    ('memory',        'abstract_noun',   'THEME',         False, True,  False),
    ('freedom',       'abstract_noun',   'GOAL',          False, False, False),
    ('beauty',        'abstract_noun',   'THEME',         False, True,  False),
]

CONCEPTS   = [d[0] for d in CONCEPT_DEFS]
POS_LABELS = [d[1] for d in CONCEPT_DEFS]
FRAME_ROLES = [d[2] for d in CONCEPT_DEFS]
CAN_AGENT  = [d[3] for d in CONCEPT_DEFS]
CAN_PATIENT = [d[4] for d in CONCEPT_DEFS]
CAN_INSTRUMENT = [d[5] for d in CONCEPT_DEFS]

print(f'Concept pool: {len(CONCEPTS)} concepts')
print('Frame role distribution:', dict(Counter(FRAME_ROLES)))
print('Agent-capable:', sum(CAN_AGENT))
print('Patient-capable:', sum(CAN_PATIENT))
print('Instrument-capable:', sum(CAN_INSTRUMENT))

## SECTION 2: Multilingual embeddings + centroids (same pattern as Exp 9)

In [ ]:
SPEAKER_WEIGHTS = {
    'en':1500, 'zh':1100, 'hi':600, 'es':560, 'ar':380,
    'ru':260, 'pt':260, 'fr':280, 'de':130, 'ja':125,
}

# 10-language vocabulary for all 56 concepts
LANG_VOCAB = {
    'en': CONCEPTS,  # English is the concept label itself
    'zh': ['水','火','土','天空','风','石头','河流','山','森林','孩子',
           '老人','朋友','敌人','时间','生命','死亡','和平','战争','希望','梦想',
           '改变','爱','恐惧','信任','喜悦','痛苦','给','拿','说话','思考',
           '找到','失去','建造','破坏','吃','感觉','睡觉','跑','光','暗',
           '教','学习','杀','生长','落下','升起','唱歌','哭',
           '刀','手','声音','家','沉默','记忆','自由','美丽'],
    'es': ['agua','fuego','tierra','cielo','viento','piedra','río','montaña','bosque','niño',
           'anciano','amigo','enemigo','tiempo','vida','muerte','paz','guerra','esperanza','sueño',
           'cambio','amor','miedo','confianza','alegría','dolor','dar','tomar','hablar','pensar',
           'encontrar','perder','construir','romper','comer','sentir','dormir','correr','luz','oscuridad',
           'enseñar','aprender','matar','crecer','caer','subir','cantar','llorar',
           'cuchillo','mano','voz','hogar','silencio','memoria','libertad','belleza'],
    'fr': ['eau','feu','terre','ciel','vent','pierre','rivière','montagne','forêt','enfant',
           'aîné','ami','ennemi','temps','vie','mort','paix','guerre','espoir','rêve',
           'changement','amour','peur','confiance','joie','douleur','donner','prendre','parler','penser',
           'trouver','perdre','construire','casser','manger','ressentir','dormir','courir','lumière','obscurité',
           'enseigner','apprendre','tuer','grandir','tomber','monter','chanter','pleurer',
           'couteau','main','voix','foyer','silence','mémoire','liberté','beauté'],
    'de': ['Wasser','Feuer','Erde','Himmel','Wind','Stein','Fluss','Berg','Wald','Kind',
           'Alter','Freund','Feind','Zeit','Leben','Tod','Frieden','Krieg','Hoffnung','Traum',
           'Veränderung','Liebe','Angst','Vertrauen','Freude','Schmerz','geben','nehmen','sprechen','denken',
           'finden','verlieren','bauen','brechen','essen','fühlen','schlafen','laufen','Licht','Dunkel',
           'lehren','lernen','töten','wachsen','fallen','steigen','singen','weinen',
           'Messer','Hand','Stimme','Zuhause','Stille','Erinnerung','Freiheit','Schönheit'],
    'pt': ['água','fogo','terra','céu','vento','pedra','rio','montanha','floresta','criança',
           'idoso','amigo','inimigo','tempo','vida','morte','paz','guerra','esperança','sonho',
           'mudança','amor','medo','confiança','alegria','dor','dar','tomar','falar','pensar',
           'encontrar','perder','construir','quebrar','comer','sentir','dormir','correr','luz','escuridão',
           'ensinar','aprender','matar','crescer','cair','subir','cantar','chorar',
           'faca','mão','voz','lar','silêncio','memória','liberdade','beleza'],
}

print(f'\nEmbedding {len(CONCEPTS)} concepts × {len(LANG_VOCAB)} languages...')
records = []
for lang, words in LANG_VOCAB.items():
    assert len(words) == len(CONCEPTS), f'Length mismatch for {lang}: {len(words)} vs {len(CONCEPTS)}'
    embs = model.encode(words, normalize_embeddings=True)
    for concept, word, emb in zip(CONCEPTS, words, embs):
        records.append({'lang': lang, 'concept': concept, 'word': word, 'emb': emb})

import pandas as pd
df = pd.DataFrame(records)

# Speaker-weighted centroids
centroids = {}
for concept in CONCEPTS:
    sub = df[df['concept'] == concept]
    w = np.array([SPEAKER_WEIGHTS[l] for l in sub['lang']], dtype=float)
    w /= w.sum()
    c = (np.stack(sub['emb'].values) * w[:, None]).sum(axis=0)
    c /= np.linalg.norm(c)
    centroids[concept] = c

centroid_matrix = np.stack([centroids[c] for c in CONCEPTS])
print(f'Centroid matrix: {centroid_matrix.shape}')

## SECTION 3: Phase 1 — Compositional projection (from Exp 9)

In [ ]:
print('\n' + '═'*60)
print('Phase 1: Training compositional projection')
print('═'*60)

# Analogy pairs for training the projection
ANALOGY_PAIRS = [
    ('love','fear','joy','pain'),
    ('life','death','peace','war'),
    ('give','take','build','break'),
    ('light','dark','hope','pain'),
    ('friend','enemy','peace','war'),
    ('child','elder','life','death'),
    ('find','lose','give','take'),
    ('teach','learn','give','take'),
    ('rise','fall','build','break'),
    ('sing','cry','joy','pain'),
]

class ProjectionLayer(nn.Module):
    def __init__(self, dim=768):
        super().__init__()
        self.W = nn.Linear(dim, dim, bias=False)
        nn.init.eye_(self.W.weight)  # start near identity

    def forward(self, x):
        out = self.W(x)
        return F.normalize(out, dim=-1)

proj_model = ProjectionLayer(768)
opt = torch.optim.Adam(proj_model.parameters(), lr=5e-4)
X_tensor = torch.tensor(centroid_matrix, dtype=torch.float32)

for epoch in range(800):
    proj_model.train()
    P = proj_model(X_tensor)

    # L_proximity: projected should stay close to original
    L_prox = 1.0 - (P * X_tensor).sum(dim=-1).mean()

    # L_compositionality: A-B+C≈D for analogy pairs
    L_comp = torch.tensor(0.0)
    n_pairs = 0
    for a, b, c, d in ANALOGY_PAIRS:
        if all(x in CONCEPTS for x in [a,b,c,d]):
            ia, ib, ic, id_ = [CONCEPTS.index(x) for x in [a,b,c,d]]
            pred = P[ia] - P[ib] + P[ic]
            pred = F.normalize(pred, dim=0)
            L_comp = L_comp + (1.0 - (pred * P[id_]).sum())
            n_pairs += 1
    L_comp = L_comp / max(n_pairs, 1)

    loss = L_prox + 2.0 * L_comp
    opt.zero_grad()
    loss.backward()
    opt.step()

    if (epoch+1) % 200 == 0:
        print(f'  Epoch {epoch+1}: L_prox={L_prox.item():.4f}  L_comp={L_comp.item():.4f}')

proj_model.eval()
with torch.no_grad():
    projected_matrix = proj_model(X_tensor).numpy()

print(f'Projected matrix: {projected_matrix.shape}')

## PROBE A: Frame Role Classification (the main test)

In [ ]:
print('\n' + '═'*60)
print('PROBE A — Frame Role Classification')
print('(Can we predict AGENT/PATIENT/THEME/EXPERIENCER/etc from geometry?)')
print('═'*60)

role_labels = np.array(FRAME_ROLES)
unique_roles = sorted(set(role_labels))
role_to_idx = {r: i for i, r in enumerate(unique_roles)}
y_role = np.array([role_to_idx[r] for r in role_labels])
n_classes = len(unique_roles)
chance = 1.0 / n_classes

print(f'Roles: {unique_roles}')
print(f'Chance baseline: {chance:.3f}')

for space_name, matrix in [('Raw LaBSE', centroid_matrix), ('Projected', projected_matrix)]:
    # LOO cross-validation with logistic regression
    loo = LeaveOneOut()
    y_pred = []
    for train_idx, test_idx in loo.split(matrix):
        clf = LogisticRegression(max_iter=2000, random_state=42, C=1.0)
        clf.fit(matrix[train_idx], y_role[train_idx])
        y_pred.append(clf.predict(matrix[test_idx])[0])
    
    acc = accuracy_score(y_role, y_pred)
    print(f'\n  {space_name}: LOO accuracy = {acc:.3f} (chance = {chance:.3f})')
    
    # Also try 5-fold CV for more stable estimate
    clf_cv = LogisticRegression(max_iter=2000, random_state=42, C=1.0)
    cv_scores = cross_val_score(clf_cv, matrix, y_role, cv=5, scoring='accuracy')
    print(f'  {space_name}: 5-fold CV accuracy = {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
    
    if acc > 2 * chance:
        print(f'  VERDICT: DETECTED — frame roles have geometric structure in {space_name}')
    elif acc > 1.5 * chance:
        print(f'  VERDICT: PARTIAL — some frame role signal in {space_name}')
    else:
        print(f'  VERDICT: NOT_DETECTED — frame roles not linearly separable in {space_name}')

## PROBE B: Continuous Role Geometry (the key insight from the suggestion)

In [ ]:
print('\n' + '═'*60)
print('PROBE B — Continuous Role Geometry')
print('(Does agent-hood, patient-hood etc. have continuous geometric signal?)')
print('═'*60)

# Instead of classifying discrete roles, train binary probes for each capability
for role_name, role_labels_binary in [
    ('can_be_agent', CAN_AGENT),
    ('can_be_patient', CAN_PATIENT),
    ('can_be_instrument', CAN_INSTRUMENT),
]:
    y_bin = np.array(role_labels_binary, dtype=int)
    n_pos = y_bin.sum()
    n_neg = len(y_bin) - n_pos
    majority_baseline = max(n_pos, n_neg) / len(y_bin)
    
    print(f'\n  {role_name}: {n_pos} positive / {n_neg} negative (majority baseline = {majority_baseline:.3f})')
    
    for space_name, matrix in [('Raw LaBSE', centroid_matrix), ('Projected', projected_matrix)]:
        loo = LeaveOneOut()
        y_pred = []
        y_proba = []
        for train_idx, test_idx in loo.split(matrix):
            clf = LogisticRegression(max_iter=2000, random_state=42, C=1.0)
            clf.fit(matrix[train_idx], y_bin[train_idx])
            y_pred.append(clf.predict(matrix[test_idx])[0])
            y_proba.append(clf.predict_proba(matrix[test_idx])[0, 1])
        
        acc = accuracy_score(y_bin, y_pred)
        # Compute role direction: mean of positive - mean of negative
        pos_centroid = matrix[y_bin == 1].mean(axis=0)
        neg_centroid = matrix[y_bin == 0].mean(axis=0)
        role_direction = pos_centroid - neg_centroid
        role_direction /= np.linalg.norm(role_direction) + 1e-9
        
        # Project all concepts onto role direction
        projections = matrix @ role_direction
        # Correlation between projection and binary label
        rho, pval = spearmanr(projections, y_bin)
        
        print(f'    {space_name}: LOO acc = {acc:.3f}  direction ρ = {rho:.3f} (p={pval:.4f})')

## PROBE C: Compositional Frame Test

In [ ]:
print('\n' + '═'*60)
print('PROBE C — Compositional Frame Test')
print('(Given a verb-noun pair, can we predict which fills which role?)')
print('═'*60)

# Create verb-noun pairs with known role assignments
FRAME_PAIRS = [
    # (verb, agent_noun, patient_noun)
    ('give',  'child', 'water'),
    ('take',  'friend', 'stone'),
    ('build', 'elder', 'home'),
    ('break', 'enemy', 'stone'),
    ('eat',   'child', 'fire'),   # "fire" as patient is odd but geometrically testable
    ('teach', 'elder', 'child'),
    ('find',  'friend', 'light'),
    ('kill',  'enemy', 'hope'),
    ('feel',  'child', 'pain'),
    ('speak', 'elder', 'voice'),  # voice as instrument
]

# For each pair: compute verb+agent vs verb+patient similarity
# Agent should be closer to verb in the "action initiator" direction

# First compute the AGENT direction
agent_concepts = [CONCEPTS[i] for i in range(len(CONCEPTS)) if CAN_AGENT[i]]
non_agent_concepts = [CONCEPTS[i] for i in range(len(CONCEPTS)) if not CAN_AGENT[i]]

for space_name, matrix in [('Raw LaBSE', centroid_matrix), ('Projected', projected_matrix)]:
    agent_vecs = np.stack([matrix[CONCEPTS.index(c)] for c in agent_concepts])
    non_agent_vecs = np.stack([matrix[CONCEPTS.index(c)] for c in non_agent_concepts])
    
    agent_dir = agent_vecs.mean(axis=0) - non_agent_vecs.mean(axis=0)
    agent_dir /= np.linalg.norm(agent_dir) + 1e-9
    
    correct = 0
    total = 0
    for verb, agent_noun, patient_noun in FRAME_PAIRS:
        if all(c in CONCEPTS for c in [verb, agent_noun, patient_noun]):
            v_idx = CONCEPTS.index(verb)
            a_idx = CONCEPTS.index(agent_noun)
            p_idx = CONCEPTS.index(patient_noun)
            
            # Project agent and patient onto agent direction
            a_proj = float(matrix[a_idx] @ agent_dir)
            p_proj = float(matrix[p_idx] @ agent_dir)
            
            # Agent should have higher projection
            if a_proj > p_proj:
                correct += 1
            total += 1
    
    acc = correct / max(total, 1)
    print(f'  {space_name}: Agent identification accuracy = {acc:.3f} ({correct}/{total})')
    if acc > 0.7:
        print(f'  → DETECTED: agent direction reliably separates agents from patients')
    elif acc > 0.55:
        print(f'  → PARTIAL: some agent/patient separation')
    else:
        print(f'  → NOT_DETECTED: no reliable agent/patient separation')

## PROBE D: Agent-Patient Asymmetry (the geometric structure test)

In [ ]:
print('\n' + '═'*60)
print('PROBE D — Agent-Patient Asymmetry')
print('(Do verb+agent pairs cluster differently from verb+patient pairs?)')
print('═'*60)

# For each verb, compute: verb_vec + agent_vec and verb_vec + patient_vec
# Then measure: are the agent-combinations more similar to each other
# than to patient-combinations?

verb_agent_vecs = []
verb_patient_vecs = []

for space_name, matrix in [('Raw LaBSE', centroid_matrix), ('Projected', projected_matrix)]:
    va_vecs, vp_vecs = [], []
    for verb, agent_noun, patient_noun in FRAME_PAIRS:
        if all(c in CONCEPTS for c in [verb, agent_noun, patient_noun]):
            v = matrix[CONCEPTS.index(verb)]
            a = matrix[CONCEPTS.index(agent_noun)]
            p = matrix[CONCEPTS.index(patient_noun)]
            
            va = v + a; va /= np.linalg.norm(va)
            vp = v + p; vp /= np.linalg.norm(vp)
            
            va_vecs.append(va)
            vp_vecs.append(vp)
    
    va_arr = np.stack(va_vecs)
    vp_arr = np.stack(vp_vecs)
    
    # Intra-group coherence vs inter-group
    intra_agent = 1 - pdist(va_arr, 'cosine').mean()
    intra_patient = 1 - pdist(vp_arr, 'cosine').mean()
    
    from sklearn.metrics.pairwise import cosine_similarity
    inter = cosine_similarity(va_arr, vp_arr)
    np.fill_diagonal(inter, 0)
    inter_mean = inter.sum() / (inter.size - len(inter))
    
    print(f'\n  {space_name}:')
    print(f'    Intra-agent coherence:  {intra_agent:.4f}')
    print(f'    Intra-patient coherence: {intra_patient:.4f}')
    print(f'    Inter-group similarity:  {inter_mean:.4f}')
    
    # Asymmetry = intra > inter means the roles create distinct geometric clusters
    asymmetry = ((intra_agent + intra_patient) / 2) - inter_mean
    print(f'    Asymmetry score: {asymmetry:.4f} (>0 = roles create distinct clusters)')

## Visualisation

In [ ]:
print('\n' + '═'*60)
print('Generating visualisations...')
print('═'*60)

fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

ROLE_COLORS = {
    'AGENT':       '#E24B4A',
    'PATIENT':     '#378ADD',
    'THEME':       '#1D9E75',
    'EXPERIENCER': '#EF9F27',
    'INSTRUMENT':  '#7F77DD',
    'GOAL':        '#D4537E',
    'STATE':       '#9E9E9E',
}

# Plot 1: PCA of raw centroids coloured by frame role
ax1 = fig.add_subplot(gs[0, 0])
pca = PCA(n_components=2)
C2 = pca.fit_transform(centroid_matrix)
for role, color in ROLE_COLORS.items():
    idx = [i for i, r in enumerate(FRAME_ROLES) if r == role]
    if idx:
        ax1.scatter(C2[idx, 0], C2[idx, 1], c=color, s=60, label=role, alpha=0.8)
for i in range(len(CONCEPTS)):
    ax1.annotate(CONCEPTS[i], (C2[i,0]+0.002, C2[i,1]+0.002), fontsize=5)
ax1.set_title('Raw LaBSE: PCA coloured by frame role', fontsize=9)
ax1.legend(fontsize=6, loc='lower left')

# Plot 2: PCA of projected centroids coloured by frame role
ax2 = fig.add_subplot(gs[0, 1])
C2p = pca.fit_transform(projected_matrix)
for role, color in ROLE_COLORS.items():
    idx = [i for i, r in enumerate(FRAME_ROLES) if r == role]
    if idx:
        ax2.scatter(C2p[idx, 0], C2p[idx, 1], c=color, s=60, label=role, alpha=0.8)
for i in range(len(CONCEPTS)):
    ax2.annotate(CONCEPTS[i], (C2p[i,0]+0.002, C2p[i,1]+0.002), fontsize=5)
ax2.set_title('Projected: PCA coloured by frame role', fontsize=9)
ax2.legend(fontsize=6, loc='lower left')

# Plot 3: Agent direction projection
ax3 = fig.add_subplot(gs[0, 2])
y_agent = np.array(CAN_AGENT, dtype=int)
agent_vecs_all = projected_matrix[y_agent == 1]
non_agent_vecs_all = projected_matrix[y_agent == 0]
agent_dir_proj = agent_vecs_all.mean(axis=0) - non_agent_vecs_all.mean(axis=0)
agent_dir_proj /= np.linalg.norm(agent_dir_proj) + 1e-9
proj_scores = projected_matrix @ agent_dir_proj
ranked = np.argsort(proj_scores)[::-1]
colors_bar = ['#E24B4A' if CAN_AGENT[i] else '#378ADD' if CAN_PATIENT[i] else '#9E9E9E' for i in ranked]
ax3.barh([CONCEPTS[i] for i in ranked], [proj_scores[i] for i in ranked],
         color=colors_bar, height=0.6)
ax3.set_xlabel('Projection onto agent direction')
ax3.set_title('Projected space: Agent direction\n(red=agent, blue=patient, grey=neither)', fontsize=8)
ax3.tick_params(labelsize=5)

# Plot 4: Binary probe results comparison
ax4 = fig.add_subplot(gs[1, 0])
# Re-run to collect numbers
probe_results = {}
for role_name, role_labels_binary in [
    ('Agent', CAN_AGENT), ('Patient', CAN_PATIENT), ('Instrument', CAN_INSTRUMENT)
]:
    y_bin = np.array(role_labels_binary, dtype=int)
    majority = max(y_bin.sum(), len(y_bin)-y_bin.sum()) / len(y_bin)
    for space_name, matrix in [('Raw', centroid_matrix), ('Projected', projected_matrix)]:
        loo = LeaveOneOut()
        y_pred = [LogisticRegression(max_iter=2000, C=1.0).fit(
            matrix[train_idx], y_bin[train_idx]
        ).predict(matrix[test_idx])[0] for train_idx, test_idx in loo.split(matrix)]
        acc = accuracy_score(y_bin, y_pred)
        probe_results[(role_name, space_name)] = acc
    probe_results[(role_name, 'chance')] = majority

roles_plot = ['Agent', 'Patient', 'Instrument']
x = np.arange(len(roles_plot))
w = 0.25
ax4.bar(x - w, [probe_results[(r, 'Raw')] for r in roles_plot], w, label='Raw LaBSE', color='#378ADD')
ax4.bar(x,     [probe_results[(r, 'Projected')] for r in roles_plot], w, label='Projected', color='#1D9E75')
ax4.bar(x + w, [probe_results[(r, 'chance')] for r in roles_plot], w, label='Majority baseline', color='#9E9E9E', alpha=0.5)
ax4.set_xticks(x)
ax4.set_xticklabels(roles_plot)
ax4.set_ylabel('LOO Accuracy')
ax4.set_title('Binary role probes: Raw vs Projected', fontsize=9)
ax4.legend(fontsize=7)
ax4.grid(axis='y', alpha=0.3)

# Plot 5: Frame role confusion matrix (projected space)
ax5 = fig.add_subplot(gs[1, 1])
clf_final = LogisticRegression(max_iter=2000, random_state=42, C=1.0)
clf_final.fit(projected_matrix, y_role)
from sklearn.metrics import confusion_matrix
y_pred_all = clf_final.predict(projected_matrix)
cm = confusion_matrix(y_role, y_pred_all)
im = ax5.imshow(cm, cmap='Blues')
ax5.set_xticks(range(n_classes))
ax5.set_yticks(range(n_classes))
ax5.set_xticklabels(unique_roles, rotation=45, ha='right', fontsize=7)
ax5.set_yticklabels(unique_roles, fontsize=7)
for i in range(n_classes):
    for j in range(n_classes):
        ax5.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=8)
ax5.set_title('Frame role confusion (projected, train set)', fontsize=9)
plt.colorbar(im, ax=ax5)

# Plot 6: Verb-noun composition (agent vs patient clusters)
ax6 = fig.add_subplot(gs[1, 2])
va_plot, vp_plot = [], []
va_labels_plot, vp_labels_plot = [], []
for verb, agent_noun, patient_noun in FRAME_PAIRS:
    if all(c in CONCEPTS for c in [verb, agent_noun, patient_noun]):
        v = projected_matrix[CONCEPTS.index(verb)]
        a = projected_matrix[CONCEPTS.index(agent_noun)]
        p = projected_matrix[CONCEPTS.index(patient_noun)]
        va = v + a; va /= np.linalg.norm(va)
        vp = v + p; vp /= np.linalg.norm(vp)
        va_plot.append(va)
        vp_plot.append(vp)
        va_labels_plot.append(f'{verb}+{agent_noun}')
        vp_labels_plot.append(f'{verb}+{patient_noun}')

all_comp = np.vstack([va_plot, vp_plot])
comp_pca = PCA(n_components=2).fit_transform(all_comp)
n_va = len(va_plot)
ax6.scatter(comp_pca[:n_va, 0], comp_pca[:n_va, 1], c='#E24B4A', s=80, label='verb+agent', alpha=0.8)
ax6.scatter(comp_pca[n_va:, 0], comp_pca[n_va:, 1], c='#378ADD', s=80, label='verb+patient', alpha=0.8)
for i, lbl in enumerate(va_labels_plot):
    ax6.annotate(lbl, (comp_pca[i,0]+0.003, comp_pca[i,1]+0.003), fontsize=5, color='#E24B4A')
for i, lbl in enumerate(vp_labels_plot):
    ax6.annotate(lbl, (comp_pca[n_va+i,0]+0.003, comp_pca[n_va+i,1]+0.003), fontsize=5, color='#378ADD')
ax6.set_title('Verb+Agent vs Verb+Patient compositions\n(PCA of summed vectors)', fontsize=9)
ax6.legend(fontsize=8)

fig.suptitle('Experiment 10 — Grammatical Frame Induction\nDo frame roles (agent/patient/theme/experiencer) have geometric structure?',
             fontsize=13, fontweight='bold')
plt.savefig('exp10_frame_induction.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print('\n' + '═'*60)
print('EXPERIMENT 10 — SUMMARY')
print('═'*60)
print('Probes:')
print('  A (Frame role classification):')
print('    → Tests multi-class role prediction (AGENT/THEME/EXPERIENCER/etc)')
print('    → If detected: roles are implicit in concept geometry')
print('  B (Continuous role geometry):')
print('    → Tests binary agent-hood, patient-hood, instrument-hood')
print('    → The key advance over Exp 9 Probe 2: continuous, not binary transitivity')
print('  C (Compositional frame test):')
print('    → Given verb+noun, can agent direction identify the agent?')
print('    → If yes: basis for Esperanto-like case suffixes grounded in data')
print('  D (Agent-patient asymmetry):')
print('    → Do verb+agent compositions cluster separately from verb+patient?')
print('    → If yes: composition itself encodes grammatical relations')
print()
print('Implication for universal language design:')
print('  DETECTED → use geometric frame roles as basis for agglutinative morphology')
print('  NOT_DETECTED → design grammatical markers explicitly (like Esperanto)')